In [1]:
import json
from sklearn.metrics import cohen_kappa_score
# Round 1
annotated_ann1 = "data_ann1_80.json"
annotated_ann2 = "data_ann2_80.json"

with open(annotated_ann1, 'r', encoding='utf-8') as file:
    ann1 = json.load(file)


with open(annotated_ann2, 'r', encoding='utf-8') as file:
    ann2 = json.load(file)

In [2]:
categories = {
    "Misread Spec": "FN_misread_spec",
    "Added Requirement": "FN_added_requirement",
    "Overthink Edge": "FN_overthink_edge",
    "Assumed Type": "FN_assumed_type",
    "Imagined Runtime": "FN_imagined_runtime",
    "Performance Nitpick": "FN_performance_nitpick",
    "Readability Nitpick": "FN_readability_nitpick",
    "Precision Error": "FN_precision_error",
    "Boundary Error": "FN_boundary_error",
    "Logic Error": "FN_logic_error",
    "Vague Description": "FN_vague_description",
    "Other": "FN_other",
}
for t in categories.values():
    a = [ann1[i][t] for i in range(len(ann1))]
    w = [ann2[i][t] for i in range(len(ann2))]

    agree = sum(x == y for x, y in zip(a, w)) / len(ann1)
    k = cohen_kappa_score(a, w)
    print(f"{t:<15} \t kappa={k:6.3f}  agreement={agree:.2%} ann1_pos={sum(a):2d}  ann2_pos={sum(w):2d}")

FN_misread_spec 	 kappa= 0.731  agreement=87.50% ann1_pos=13  ann2_pos=16
FN_added_requirement 	 kappa= 0.595  agreement=82.50% ann1_pos=14  ann2_pos=11
FN_overthink_edge 	 kappa= 0.531  agreement=92.50% ann1_pos= 4  ann2_pos= 3
FN_assumed_type 	 kappa=   nan  agreement=100.00% ann1_pos= 0  ann2_pos= 0
FN_imagined_runtime 	 kappa=   nan  agreement=100.00% ann1_pos= 0  ann2_pos= 0
FN_performance_nitpick 	 kappa=   nan  agreement=100.00% ann1_pos= 0  ann2_pos= 0
FN_readability_nitpick 	 kappa=   nan  agreement=100.00% ann1_pos= 0  ann2_pos= 0
FN_precision_error 	 kappa=   nan  agreement=100.00% ann1_pos= 0  ann2_pos= 0
FN_boundary_error 	 kappa=   nan  agreement=100.00% ann1_pos= 0  ann2_pos= 0
FN_logic_error  	 kappa= 0.776  agreement=92.50% ann1_pos= 8  ann2_pos= 9
FN_vague_description 	 kappa= 1.000  agreement=100.00% ann1_pos= 1  ann2_pos= 1
FN_other        	 kappa=   nan  agreement=100.00% ann1_pos= 0  ann2_pos= 0


/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/sklearn/metrics/_classification.py:897: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/sklearn/metrics/_classification.py:897: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expect

In [3]:
ann1_by_id = {r["id"]: r for r in ann1}
ann2_by_id = {r["id"]: r for r in ann2}
ids = sorted(set(ann1_by_id) & set(ann2_by_id))

a = []
w = []
disagreements = {}
for i in ids:
    ann1_labels = [label for label, t in categories.items() if ann1_by_id[i][t]]
    ann2_labels = [label for label, t in categories.items() if ann2_by_id[i][t]]
    
    if len(ann1_labels) != 1:
        print('ann1 duplicate')
        print(f"id: {i}")
        print(ann1_labels)
        print(f"requirement: {ann1_by_id[i]['description']}")
        print()
    if len(ann2_labels) != 1: 
        print('ann2 duplicate')
        print(f"id: {i}")
        print(ann2_labels)
        print(f"requirement: {ann2_by_id[i]['description']}")
    
    ann1_labels = ann1_labels[0]
    ann2_labels = ann2_labels[0]
    if ann1_labels != ann2_labels:
        disagreements[i] = (ann1_labels, ann2_labels)
        
    a.append(ann1_labels)
    w.append(ann2_labels)


In [4]:
k = cohen_kappa_score(a, w)
k

0.6877710320901995

In [5]:
for i, (ann1_label, ann2_label) in disagreements.items():
    print(f"id: {i}")
    print(f"requirement: {ann1_by_id[i]['description']}")
    print(f"ann1: {ann1_label}")
    print(f"ann2: {ann2_label}")
    print()


id: 49
requirement: The system may allow a covered entity to obtain an individual’s consent to use or disclose protected health information for payment operations.
ann1: Added Requirement
ann2: Misread Spec

id: 51
requirement: The system shall ensure that consent is not treated as valid for use or disclosure of protected health information when an authorization is required.
ann1: Logic Error
ann2: Added Requirement

id: 60
requirement: The system shall ensure the integrity of all electronic protected health information (ePHI) that it maintains.
ann1: Added Requirement
ann2: Logic Error

id: 63
requirement: The system shall support implementing security measures sufficient to reduce risks and vulnerabilities to a reasonable and appropriate level to comply with the general security requirements.
ann1: Added Requirement
ann2: Overthink Edge

id: 64
requirement: The system shall support procedures to regularly review records of information system activity, including audit logs, access rep

In [6]:
for i in ids:
    diffs = [t for t in categories.values() if bool(ann1_by_id[i][t]) != bool(ann2_by_id[i][t])]
    if not diffs:
        continue
    print(f"id: {i}")
    print(f"requirement: {ann1_by_id[i]['description']}")
    for t in diffs:
        who = "ann1" if ann1_by_id[i][t] else "ann2"
        print(f"  - {t}: True by {who}")
    for name, rec in (("ann1", ann1_by_id[i]), ("ann2", ann2_by_id[i])):
        note = (rec.get("notes") or rec.get("note") or "").strip()
        if note:
            print(f"  note ({name}): {note}")
    print()

id: 49
requirement: The system may allow a covered entity to obtain an individual’s consent to use or disclose protected health information for payment operations.
  - FN_misread_spec: True by ann2
  - FN_added_requirement: True by ann1

id: 51
requirement: The system shall ensure that consent is not treated as valid for use or disclosure of protected health information when an authorization is required.
  - FN_added_requirement: True by ann2
  - FN_logic_error: True by ann1

id: 60
requirement: The system shall ensure the integrity of all electronic protected health information (ePHI) that it maintains.
  - FN_added_requirement: True by ann1
  - FN_logic_error: True by ann2

id: 63
requirement: The system shall support implementing security measures sufficient to reduce risks and vulnerabilities to a reasonable and appropriate level to comply with the general security requirements.
  - FN_added_requirement: True by ann1
  - FN_overthink_edge: True by ann2

id: 64
requirement: The syst

# self consistency

In [7]:
categories = {
    "Consistent": "self_consistency_consistent",
    "Contradiction": "self_consistency_contradiction",
    "Unclear": "self_consistency_unclear",
}
for t in categories.values():
    a = [ann1[i][t] for i in range(len(ann1))]
    w = [ann2[i][t] for i in range(len(ann2))]

    agree = sum(x == y for x, y in zip(a, w)) / len(ann1)
    k = cohen_kappa_score(a, w)
    print(f"{t:<15} \t kappa={k:6.3f}  agreement={agree:.2%} ann1_pos={sum(a):2d}  ann2_pos={sum(w):2d}")

self_consistency_consistent 	 kappa= 0.000  agreement=0.00% ann1_pos= 0  ann2_pos=40
self_consistency_contradiction 	 kappa=   nan  agreement=100.00% ann1_pos= 0  ann2_pos= 0
self_consistency_unclear 	 kappa=   nan  agreement=100.00% ann1_pos= 0  ann2_pos= 0


/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/sklearn/metrics/_classification.py:897: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/miniconda3/envs/dialogue/lib/python3.10/site-packages/sklearn/metrics/_classification.py:897: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expect